# LAI Data: Merge, Clean & Vegetation-Filter Pipeline

**Purpose:** Combine Leaf Area Index (LAI) data from two time periods into a single
dataset, validate grid alignment with the weather data, and spatially filter to only
grid cells with combustible vegetation.

**Pipeline Overview:**

| Phase | Steps |
|-------|-------|
| **A. Merge Time Periods** | Load extended LAI (1994–2000) and main LAI (2001–2021), align time coordinates, concatenate, save as `.parquet` |
| **B. Grid Validation** | Confirm LAI lon/lat grid matches the weather grid |
| **C. Vegetation Filter** | Exclude Water/Urban/Agriculture grid cells, save final filtered `.parquet` |

**Inputs:**

| Data | Path |
|------|------|
| Extended LAI NetCDF (1994–2000) | `E:/zcao/CA_Wildfire/Extended_Data/LAI_Data/LAI.1994.2000.CA.daily_interpolated.nc` |
| Main LAI NetCDF (2001–2021) | `E:/zcao/CA_Wildfire/LAI_Data/glass.lai.2001.2021.CA.daily_interpolated.nc` |
| Weather parquet (for grid check) | `<clean_dir>/Weather_Data/Extended_Weather_Data/wind_speed.parquet` |
| Vegetation grid (from Veg pipeline) | `<clean_dir>/lon_lat_pair_weather_match_veg_v2.parquet` |

**Outputs:**

| Data | Path |
|------|------|
| Combined LAI parquet | `<clean_dir>/Extended_Data/Vegetation/LAI.parquet` |
| Veg-filtered LAI parquet | `<clean_dir>/Extended_Data_w_Veg_Filter/Vegetation/LAI.parquet` |

## 0. Configuration

Centralized path configuration — **edit this cell only** to adapt to your local setup.

In [2]:
import os

# ====================== EDIT THESE PATHS ======================
PROJECT_ROOT = r"E:\zcao\CA_Wildfire"

# Input paths
EXTENDED_LAI_NC = os.path.join(
    PROJECT_ROOT, "Extended_Data", "LAI_Data",
    "LAI.1994.2000.CA.daily_interpolated.nc"
)
MAIN_LAI_NC = os.path.join(
    PROJECT_ROOT, "LAI_Data",
    "glass.lai.2001.2021.CA.daily_interpolated.nc"
)

# Clean / output root
CLEAN_DIR = os.path.join(PROJECT_ROOT, "Clean_Data")

# Derived paths (auto-created)
COMBINED_LAI_DIR   = os.path.join(CLEAN_DIR, "Extended_Data", "Vegetation")
FILTERED_LAI_DIR   = os.path.join(CLEAN_DIR, "Extended_Data_w_Veg_Filter", "Vegetation")
WEATHER_REF_PATH   = os.path.join(CLEAN_DIR, "Weather_Data", "Extended_Weather_Data", "wind_speed.parquet")
VEG_V2_PATH        = os.path.join(CLEAN_DIR, "Veg_Data", "lon_lat_pair_weather_match_veg_v2.parquet")
# ==============================================================

for d in [COMBINED_LAI_DIR, FILTERED_LAI_DIR]:
    os.makedirs(d, exist_ok=True)

print("Input paths:")
for label, path in [("Extended LAI .nc", EXTENDED_LAI_NC),
                     ("Main LAI .nc", MAIN_LAI_NC),
                     ("Weather ref parquet", WEATHER_REF_PATH),
                     ("Veg v2 parquet", VEG_V2_PATH)]:
    exists = os.path.exists(path)
    status = "\u2713" if exists else "\u2717 NOT FOUND"
    print(f"  {status}  {label}: {path}")
print(f"\nOutput dirs:")
print(f"  Combined:  {COMBINED_LAI_DIR}")
print(f"  Filtered:  {FILTERED_LAI_DIR}")

Input paths:
  ✓  Extended LAI .nc: E:\zcao\CA_Wildfire\Extended_Data\LAI_Data\LAI.1994.2000.CA.daily_interpolated.nc
  ✓  Main LAI .nc: E:\zcao\CA_Wildfire\LAI_Data\glass.lai.2001.2021.CA.daily_interpolated.nc
  ✓  Weather ref parquet: E:\zcao\CA_Wildfire\Clean_Data\Weather_Data\Extended_Weather_Data\wind_speed.parquet
  ✓  Veg v2 parquet: E:\zcao\CA_Wildfire\Clean_Data\Veg_Data\lon_lat_pair_weather_match_veg_v2.parquet

Output dirs:
  Combined:  E:\zcao\CA_Wildfire\Clean_Data\Extended_Data\Vegetation
  Filtered:  E:\zcao\CA_Wildfire\Clean_Data\Extended_Data_w_Veg_Filter\Vegetation


## 1. Environment Setup

In [3]:
import sys
import gc
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from datetime import datetime
from tqdm import tqdm

print(f"Python  : {sys.version.split(chr(124))[0].strip()}")
print(f"pandas  : {pd.__version__}")
print(f"xarray  : {xr.__version__}")
print(f"numpy   : {np.__version__}")

Python  : 3.9.13 (main, Aug 25 2022, 23:51:50) [MSC v.1916 64 bit (AMD64)]
pandas  : 2.2.2
xarray  : 2024.7.0
numpy   : 1.24.4


---

# Phase A: Merge Extended + Main LAI Data

Load both LAI NetCDF files, align their time coordinates and columns,
then concatenate into a single DataFrame covering 1994–2021.

## 2. Inspect Extended LAI (1994–2000)

In [4]:
ds_ext = xr.open_dataset(EXTENDED_LAI_NC)

print(f"Dimensions : {dict(ds_ext.dims)}")
print(f"Time range : {str(ds_ext.coords['time'].min().values)[:10]} to {str(ds_ext.coords['time'].max().values)[:10]}")
print(f"Coordinates: {list(ds_ext.coords)}")
print(f"Variables  : {list(ds_ext.data_vars)}")

Dimensions : {'time': 2558, 'lat': 238, 'lon': 258}
Time range : 1994-01-01 to 2001-01-01
Coordinates: ['time', 'lat', 'lon']
Variables  : ['LAI']


## 3. Inspect Main LAI (2001–2021)

The main LAI file stores time as integer `YYYYMMDD` — we need to parse it to `datetime`.

In [5]:
ds_main = xr.open_dataset(MAIN_LAI_NC)

# Parse integer time codes (e.g. 20010101) to datetime
ds_main = ds_main.assign_coords(
    time=pd.to_datetime(ds_main["time"].values.astype(int).astype(str), format="%Y%m%d")
)

print(f"Dimensions : {dict(ds_main.dims)}")
print(f"Time range : {str(ds_main.coords['time'].min().values)[:10]} to {str(ds_main.coords['time'].max().values)[:10]}")
print(f"Coordinates: {list(ds_main.coords)}")
print(f"Variables  : {list(ds_main.data_vars)}")

Dimensions : {'lon': 259, 'lat': 240, 'time': 7666}
Time range : 2001-01-01 to 2021-12-27
Coordinates: ['lon', 'lat', 'time']
Variables  : ['LAI']


## 4. Align & Concatenate

- Trim the extended dataset to end at 2000-12-31 (avoid overlap with main data's 2001-01-01)
- Verify spatial grid alignment between the two sources
- Keep only the shared columns: `time`, `lat`, `lon`, `LAI`

In [6]:
# Trim extended data to avoid overlap
ds_ext = ds_ext.sel(time=slice("1994-01-01", "2000-12-31"))
print(f"Extended after trim: {str(ds_ext.coords['time'].min().values)[:10]} to {str(ds_ext.coords['time'].max().values)[:10]}")

# Check spatial grid overlap
common_lon = np.intersect1d(ds_ext.coords["lon"].values, ds_main.coords["lon"].values)
common_lat = np.intersect1d(ds_ext.coords["lat"].values, ds_main.coords["lat"].values)
print(f"Common grid: {len(common_lon)} lon x {len(common_lat)} lat")

Extended after trim: 1994-01-01 to 2000-12-31
Common grid: 257 lon x 233 lat


In [7]:
# Convert to DataFrames, keep only needed columns
COLS = ["time", "lat", "lon", "LAI"]

lai_ext  = ds_ext.to_dataframe().reset_index()[COLS]
lai_main = ds_main.to_dataframe().reset_index()[COLS]

print(f"Extended:  {lai_ext.shape[0]:,} rows")
print(f"Main:      {lai_main.shape[0]:,} rows")

# Concatenate
lai_combined = pd.concat([lai_ext, lai_main], ignore_index=True)

# Sanity check: row count should be exact sum
assert lai_combined.shape[0] == lai_ext.shape[0] + lai_main.shape[0], "Row count mismatch!"
print(f"Combined:  {lai_combined.shape[0]:,} rows")

# Free memory
del ds_ext, ds_main, lai_ext, lai_main
gc.collect()

Extended:  157,010,028 rows
Main:      476,518,560 rows
Combined:  633,528,588 rows


32

### Save Combined LAI

In [8]:
combined_path = os.path.join(COMBINED_LAI_DIR, "LAI.parquet")
lai_combined.to_parquet(combined_path, index=False)

print(f"Saved to: {combined_path}")
print(f"File size: {os.path.getsize(combined_path) / 1e6:.1f} MB")

Saved to: E:\zcao\CA_Wildfire\Clean_Data\Extended_Data\Vegetation\LAI.parquet
File size: 2231.0 MB


### Missing Rate by Year

Quick QA: check that LAI missing rates are consistent across the two time periods.

In [9]:
lai_combined["year"] = lai_combined["time"].dt.year
missing_by_year = lai_combined.groupby("year")["LAI"].apply(lambda x: x.isnull().mean())

print("LAI missing rate by year:")
print(missing_by_year.to_string())

LAI missing rate by year:
year
1994    0.626067
1995    0.626067
1996    0.626067
1997    0.626067
1998    0.626067
1999    0.626067
2000    0.626067
2001    0.673845
2002    0.681087
2003    0.670824
2004    0.668499
2005    0.651382
2006    0.671438
2007    0.680602
2008    0.675228
2009    0.673124
2010    0.669819
2011    0.665930
2012    0.677213
2013    0.679982
2014    0.678052
2015    0.675516
2016    0.676242
2017    0.672758
2018    0.679234
2019    0.668661
2020    0.665746
2021    0.679524


---

# Phase B: Grid Validation

Confirm that the LAI lon/lat grid is a superset of (or matches) the weather grid.
This ensures the downstream merge won't lose grid cells.

## 5. Compare LAI Grid vs. Weather Grid

In [10]:
# Load a reference weather file to get the grid
weather_ref = pd.read_parquet(WEATHER_REF_PATH)
weather_ref = weather_ref[weather_ref["day"] == "1998-01-01"]
weather_grid = weather_ref[["lon", "lat"]].drop_duplicates()
print(f"Weather grid cells: {len(weather_grid):,}")

# LAI grid
lai_grid = lai_combined[["lon", "lat"]].drop_duplicates()
print(f"LAI grid cells:     {len(lai_grid):,}")

# Inner join to check overlap
overlap = lai_grid.merge(weather_grid, on=["lon", "lat"], how="inner")
print(f"Overlapping cells:  {len(overlap):,}")

if len(overlap) == len(weather_grid):
    print("\n\u2713 LAI grid fully covers the weather grid.")
else:
    missing = len(weather_grid) - len(overlap)
    print(f"\n\u2717 WARNING: {missing} weather grid cells not found in LAI grid.")

del weather_ref, weather_grid, lai_grid, overlap
gc.collect()

Weather grid cells: 61,404
LAI grid cells:     63,683
Overlapping cells:  61,404

✓ LAI grid fully covers the weather grid.


0

---

# Phase C: Vegetation Filter

Spatially filter the combined LAI data to only include grid cells with
combustible vegetation — excluding Water, Urban, and Agriculture types.

## 6. Load & Filter Vegetation Grid

In [11]:
veg_data = pd.read_parquet(VEG_V2_PATH)
print(f"Vegetation grid: {veg_data.shape[0]:,} rows")

veg_filtered = veg_data[~veg_data["veg"].str.contains("Water|Urban|Agriculture")]
filter_keys = veg_filtered[["lon", "lat"]].drop_duplicates()

print(f"After filter:    {len(filter_keys):,} grid cells")
print(f"Removed:         {veg_data[['lon','lat']].drop_duplicates().shape[0] - len(filter_keys):,} (Water/Urban/Agriculture)")

del veg_data, veg_filtered
gc.collect()

Vegetation grid: 17,714 rows
After filter:    14,394 grid cells
Removed:         3,320 (Water/Urban/Agriculture)


25

## 7. Apply Vegetation Filter to LAI Data

In [12]:
print(f"Loading combined LAI from: {combined_path}")
lai = pd.read_parquet(combined_path)
print(f"Before filter: {lai.shape[0]:,} rows")

lai_filtered = lai.merge(filter_keys, on=["lon", "lat"], how="inner")
print(f"After filter:  {lai_filtered.shape[0]:,} rows")
print(f"Reduction:     {(1 - lai_filtered.shape[0]/lai.shape[0])*100:.1f}%")

del lai
gc.collect()

Loading combined LAI from: E:\zcao\CA_Wildfire\Clean_Data\Extended_Data\Vegetation\LAI.parquet
Before filter: 633,528,588 rows
After filter:  147,065,536 rows
Reduction:     76.8%


0

### Save Filtered LAI

In [13]:
filtered_path = os.path.join(FILTERED_LAI_DIR, "LAI.parquet")
lai_filtered.to_parquet(filtered_path, index=False)

print(f"Saved to: {filtered_path}")
print(f"File size: {os.path.getsize(filtered_path) / 1e6:.1f} MB")

Saved to: E:\zcao\CA_Wildfire\Clean_Data\Extended_Data_w_Veg_Filter\Vegetation\LAI.parquet
File size: 1080.0 MB


### Filtered LAI Missing Rate by Year

In [14]:
lai_filtered["year"] = lai_filtered["time"].dt.year
missing_by_year = lai_filtered.groupby("year")["LAI"].apply(lambda x: x.isnull().mean())

print("LAI missing rate by year (after veg filter):")
print(missing_by_year.to_string())

LAI missing rate by year (after veg filter):
year
1994    0.078227
1995    0.078227
1996    0.078227
1997    0.078227
1998    0.078227
1999    0.078227
2000    0.078227
2001    0.053565
2002    0.050510
2003    0.048234
2004    0.050983
2005    0.051144
2006    0.048921
2007    0.050153
2008    0.053587
2009    0.049206
2010    0.049184
2011    0.045130
2012    0.046274
2013    0.048790
2014    0.046118
2015    0.043975
2016    0.046573
2017    0.048877
2018    0.047935
2019    0.049485
2020    0.043612
2021    0.046295


## 8. Summary

| Phase | Step | Description | Key Result |
|-------|------|-------------|------------|
| A | Load extended | `LAI.1994.2000.CA.daily_interpolated.nc` | 1994–2000 |
| A | Load main | `glass.lai.2001.2021.CA.daily_interpolated.nc` (parse int time) | 2001–2021 |
| A | Trim & concat | Remove overlap at 2001-01-01, union both | Full 1994–2021 |
| A | Save | `LAI.parquet` | Combined output |
| B | Grid check | Confirm LAI grid covers weather grid | Inner join validation |
| C | Veg filter | Exclude Water/Urban/Agriculture | Combustible cells only |
| C | Save | `LAI.parquet` (filtered) | Final output |

**Next steps:** Merge the veg-filtered LAI data with fire ignition and weather
datasets to build the full feature matrix for ignition-risk modeling.